# Experiment 1 - Cost of Basic Operations

Times encryption, addition, multiplication, summation, average, dot product and decryption in plaintext, TenSEAL and OpenFHE, and reports each encrypted operation as a multiple of its plaintext equivalent.

Also times key generation, and measures memory, ciphertext size and approximation error.

Corresponds to Experiment 1 in the report, *Results: The Cost of Basic Operations*.

## 1. Install and load the project


In [ ]:
import subprocess
import sys

# TenSEAL publishes real platform-tagged wheels (win_amd64 / manylinux / macosx,
# CPython 3.8-3.12), so it installs anywhere. Pinned to the version the report used.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tenseal==0.3.15", "numpy", "matplotlib"], check=True)

# OpenFHE is the opposite. Every OpenFHE distribution on PyPI is tagged
# `py3-none-any`, but the payload is Linux ELF binaries built against one
# specific CPython. The 1.4.1 line, which the report used, is:
#
#     openfhe==1.4.1.0.20.4   Ubuntu 20.04   cpython-38    requires_python >=3.8
#     openfhe==1.4.1.0.22.4   Ubuntu 22.04   cpython-310   requires_python >=3.10
#     openfhe==1.4.1.0.24.4   Ubuntu 24.04   cpython-312   requires_python >=3.12
#
# Those bounds are `>=`, so an unpinned install takes the newest wheel whose
# bound this interpreter satisfies -- which need not be built for it. On 3.10
# and 3.12 that happens to land on a valid build; on 3.9, 3.11 and 3.13 it does
# not (3.11 resolves to the cpython-310 wheel openfhe==1.5.1.0.22.4), installs
# with no warning, and fails at *import*. Pinning also keeps the OpenFHE version
# identical to the report's.
OPENFHE_WHEELS = {(3, 8): "1.4.1.0.20.4", (3, 10): "1.4.1.0.22.4", (3, 12): "1.4.1.0.24.4"}
_py = sys.version_info[:2]
_wheel = OPENFHE_WHEELS.get(_py)

if sys.platform != "linux":
    print(f"Skipping OpenFHE: no wheel has ever been published for {sys.platform!r}. "
          "The plaintext and TenSEAL columns still run.")
elif _wheel is None:
    _have = ", ".join(f"{a}.{b}" for a, b in sorted(OPENFHE_WHEELS))
    print(f"Skipping OpenFHE: no build exists for CPython {_py[0]}.{_py[1]} "
          f"(builds exist for {_have}). The plaintext and TenSEAL columns still run.")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"openfhe=={_wheel}"], check=True)
    print(f"Installed openfhe=={_wheel} for CPython {_py[0]}.{_py[1]}.")

In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/To2004/confidential-computing-project.git"

# Works both on a fresh Colab runtime and inside a local checkout.
if not os.path.exists("src/benchmark_harness.py"):
    if not os.path.exists("confidential-computing-project"):
        subprocess.run(["git", "clone", "-q", REPO], check=True)
    os.chdir("confidential-computing-project")

# Absolute, so imports survive a later change of directory, and guarded so
# re-running this cell does not stack duplicate entries.
SRC = os.path.abspath("src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# Write this notebook's output to its own directory, so running it does not
# overwrite the results and figures the report was built from.
import benchmark_harness as harness
import project_paths

os.makedirs("notebook_output", exist_ok=True)
project_paths.RESULTS_DIR = "notebook_output"
project_paths.FIGURES_DIR = "notebook_output"

print("working directory:", os.getcwd())
print("report uses", harness.DEFAULT_REPEATS, "repetitions per measurement")

## 2. Check which libraries loaded

OpenFHE only publishes Linux binaries, and each build targets one specific
CPython (3.8, 3.10 or 3.12). On any other platform or Python version the
install cell above skips it, so it will not be listed as available here.

Without OpenFHE the experiment still runs, with the plaintext and TenSEAL columns only.

In [ ]:
import importlib

for name in ("tenseal", "openfhe"):
    try:
        importlib.import_module(name)
        print(f"{name}: available")
    except Exception as exc:
        print(f"{name}: not available -- {exc}")

## 3. The measurement instrument

Every number in this experiment - plaintext, TenSEAL and OpenFHE alike - comes out of one
shared module, `src/benchmark_harness.py`. The three columns of every table below differ
*only* in the call being timed: same clock, same warm-up policy, same statistics, same drift
check.

The cells in this section display the real source, read out of `src/` at run time, so a
snippet shown here cannot drift from the code that produced the numbers.

In [ ]:
import notebook_tools as nt

nt.show_source(
    "benchmark_harness.py", "time_operation",
    note="Four decisions in twelve lines: warm-up calls are discarded; GC is disabled inside "
         "the measured loop, as CPython's own `timeit` does, so an unrelated collection is "
         "never charged to the operation under test; `perf_counter` wraps a *single* call "
         "rather than a batch; and GC is restored in a `finally`, so the process is never "
         "left with collection switched off.")

In [ ]:
nt.show_source(
    "benchmark_harness.py", "assert_accuracy",
    note="Correctness gates the timing. Every decrypted result is compared against the "
         "plaintext computation of the same values, and a misconfigured run raises here "
         "rather than reporting fast, wrong numbers.")

### One harness, three backends

Each backend hands the harness a dict of zero-argument closures, and the harness never learns
which library it is timing. Every closure rebuilds its result from the same fresh operands, so
no repetition inherits accumulated noise or a consumed multiplicative level from the one before
it - call 1000 does identical work to call 1.

In [ ]:
for filename in ("plaintext_baseline.py", "tenseal_benchmark.py", "openfhe_benchmark.py"):
    nt.show_assignment(filename, "run_benchmark", "operations")

## 4. Run the experiment

`REPEATS` is the number of timed repetitions per measurement. The report uses 1000 on a reserved compute node; this notebook uses fewer so it finishes in a few minutes. The numbers it prints will therefore be noisier than the ones in the report.

In [ ]:
# 200 rather than a smaller number for one specific reason: the batch-means
# confidence interval needs at least 25 x 4 = 100 samples, and below that the
# harness falls back to the naive i.i.d. formula. Section 6 compares the two.
REPEATS = 200

!"{sys.executable}" src/run_comparison.py --repeats {REPEATS} --warmup 10 --keygen-repeats 5 --output notebook_output/results.json

## 5. Results


In [ ]:
import plot_results as pr
from IPython.display import Image, display

data = pr.load("results.json")
pr.apply_style()
pr.chart_operations(data)
pr.chart_errors(data)

display(Image("notebook_output/chart_operations.png"))
display(Image("notebook_output/chart_errors.png"))

## 6. Are these numbers trustworthy?

A confidence interval measures how consistent the samples are *with each other*, assuming they
are independent draws from one distribution. Back-to-back timings are **not** independent -
they share cache, CPU frequency and scheduler state - and if the machine changes underneath the
benchmark, the interval does not notice. It gets narrower, not wider.

The harness therefore does two things beyond reporting a mean.

In [ ]:
nt.show_source(
    "benchmark_harness.py", "_batch_means_half_width_ms",
    note="Intervals come from the means of contiguous blocks, which are much closer to "
         "independent than the raw samples are. On the report's run the naive i.i.d. formula "
         "would have reported a half-width roughly six times narrower - an artefact of the "
         "independence assumption, not a property of the measurement.")

In [ ]:
nt.show_sources([
    ("benchmark_harness.py", "_drift_diagnostics",
     "Splits the acquisition-ordered samples in half and compares the two means, so a machine "
     "that changed mid-run is flagged rather than averaged."),
    ("benchmark_harness.py", "_mann_kendall_z",
     "A halves comparison misses a slow ramp - thermal throttling, allocator growth. This "
     "rank-based trend test catches it: a synthetic +10% linear ramp shows only ~5% "
     "halves-drift but a trend z-score of 27.3."),
])

### The diagnostics, applied to the run above

This reads the JSON the experiment just wrote and reports, per operation: relative standard
deviation, the naive i.i.d. interval, the batch-means interval actually used, how much wider
that is, the lag-one autocorrelation that justifies using it, and both drift tests.

In [ ]:
import json

with open(project_paths.result_path("results.json")) as handle:
    measured = json.load(handle)

def pct(value, mean):
    return value / mean * 100.0 if mean else float("nan")

header = (f"{'operation':<12}{'rel sd %':>10}{'CI iid %':>10}{'CI batch %':>12}"
          f"{'wider by':>10}{'lag-1 r':>9}{'drift %':>9}{'trend z':>9}   flag")
print(header)
print("-" * len(header))

methods = set()
for column in ("plaintext", "tenseal", "openfhe"):
    results = measured.get(column)
    if not results:
        print(f"{column:<12}unavailable on this runtime")
        continue
    print(f"{column}:")
    for key in ("add", "mul", "sum", "avg", "dot"):
        stats = results.get(f"{key}_time_stats")
        if stats is None:
            continue
        mean = stats["mean_ms"]
        inflation = stats.get("ci95_inflation_vs_iid")
        methods.add(stats.get("ci95_method", "unknown"))
        widened = f"{inflation:.1f}x" if inflation else "n/a"
        print(f"  {key:<10}{stats['rel_std_pct']:>10.1f}"
              f"{pct(stats['ci95_iid_half_width_ms'], mean):>10.2f}"
              f"{pct(stats['ci95_half_width_ms'], mean):>12.2f}"
              f"{widened:>10}"
              f"{stats.get('lag1_autocorrelation', 0.0):>9.2f}"
              f"{(stats.get('drift_pct') or 0.0):>9.1f}"
              f"{stats.get('trend_z', 0.0):>9.1f}"
              f"   {'FLAGGED' if stats.get('drift_flagged') else ''}")

print()
for method in sorted(methods):
    print(f"interval method: {method}")
print()
print("'wider by' is the batch-means half-width divided by the i.i.d. one.")
print("Above 1x means the samples are correlated, so the i.i.d. interval")
print("would have overstated precision. 'n/a' means too few samples to batch.")

**How to read this.** On the reserved node the report used, every encrypted measurement had a
batch-means interval within 1.4% of its mean and **nothing was drift-flagged** - the single
flagged measurement in the entire published run was the *plaintext* dot product, at -5.1%.

On a laptop or a hosted runtime, expect the opposite: flagged rows here are the check working,
not a bug. Competing load cannot be averaged away, and the fix is not more repetitions - it is
not sharing the machine. That is why the published numbers were produced on an exclusive SLURM
node with the thread count pinned to one (`submit_benchmarks.sbatch`); moving off a shared
login node cut worst-case drift on the encrypted measurements from 30.9% to 2.9%.